# Kalshi Whale Tracker
**Team:** Seattle Sonics  
**Course:** CS 122 — Advanced Python Programming  
**Assignment:** Project Assignment 02

This notebook identifies and profiles high-volume traders ("Whales") in the Kalshi Economics market.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from indexers.kalshi_loader import load_kalshi_trades, get_schema
from analysis.whale_identifier import identify_whales, compute_market_concentration

OUTPUT_DIR = Path.cwd().parent / 'output'
DATA_DIR = Path.cwd().parent / 'data'
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

## 2. Dataset Overview

Before loading the full dataset, inspect its schema to understand available columns.

In [ ]:
# Update this path to point to your Parquet file/directory
PARQUET_PATH = DATA_DIR / 'kalshi'  # or the full path to the .parquet file

if PARQUET_PATH.exists():
    print('Schema:')
    get_schema(PARQUET_PATH)
else:
    print(f'Data not found at {PARQUET_PATH}')
    print('Place the Kalshi Parquet files in the data/ directory.')

## 3. Load & Filter Kalshi Economics Trades

In [ ]:
trades = load_kalshi_trades(PARQUET_PATH)

print(f'Rows loaded: {len(trades):,}')
print(f'Columns: {list(trades.columns)}')
trades.head()

## 4. Statistical Profile

In [ ]:
trades.describe()

## 5. Identify Whales

Whale definition: top 0.5% by total volume, minimum $5,000 per trade.

In [ ]:
# Update column names to match your actual dataset schema
TRADER_COL = 'trader_id'   # column identifying the trader
VOLUME_COL  = 'volume'     # column for trade size/volume

whales = identify_whales(trades, trader_col=TRADER_COL, volume_col=VOLUME_COL)
concentration = compute_market_concentration(trades, whales, trader_col=TRADER_COL, volume_col=VOLUME_COL)

print(f"Whales identified: {concentration['num_whales']}")
print(f"Whale volume share: {concentration['whale_concentration_pct']:.1f}%")
whales.head(10)

## 6. Visualizations

In [ ]:
# --- Figure 1: Whale Volume Concentration (Pie Chart) ---
fig, ax = plt.subplots(figsize=(7, 7))
labels = ['Whales (Top 50)', 'Other Traders']
sizes  = [
    concentration['whale_volume'],
    concentration['total_market_volume'] - concentration['whale_volume'],
]
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140,
       colors=['#e07b54', '#5b8db8'])
ax.set_title('Whale Volume Concentration — Kalshi Economics Market')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'whale_concentration.png', dpi=150)
plt.show()

In [ ]:
# --- Figure 2: Top 50 Whales by Total Volume (Bar Chart) ---
fig, ax = plt.subplots(figsize=(14, 6))
top = whales.head(50)
ax.bar(range(len(top)), top['total_volume'], color='#e07b54')
ax.set_xlabel('Whale Rank')
ax.set_ylabel('Total Volume ($)')
ax.set_title('Top 50 Whale Traders — Total Volume')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top50_whale_volume.png', dpi=150)
plt.show()

## 7. Save Summary CSV

In [ ]:
whales.to_csv(OUTPUT_DIR / 'whale_summary.csv', index=False)
print(f'Saved whale_summary.csv with {len(whales)} rows.')